# 04 · Загвар: LightGBM, XGBoost, CatBoost + Optuna

Гурван gradient boosting алгоритмыг **ижил нөхцөлд** сургаж, аль нь хамгийн сайныг **validation**-аар сонгоно.

| Юу | Тохиргоо (өөрчлөгдөхгүй) |
|---|---|
| Feature | 18 (`feature_list.json`, `category` хассан — ADR 0005) |
| Target | Δ = y_h − y_0 (y = log1p(анхаарал)), h = 1, 2, 3 тус бүрт тусдаа model |
| Loss | L1 (MAE) — медианыг таамагладаг тул вирал үсрэлтэд тэсвэртэй |
| Хуваалт | train ≤ 2025-06-30 < valid ≤ 2026-03-31 < test |

**Дүрэм:** сонголт зөвхөн **(в)-гүй valid MAE**-ээр. **Test-ийг энд огт ашиглахгүй** (дараагийн 05 алхамд нэг удаа).

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go, plotly.io as pio
from plotly.subplots import make_subplots
import optuna
warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"
BLUE, RED, GRAY = "#2a78d6", "#e63946", "#9aa0ad"
COLORS = {"lightgbm": BLUE, "xgboost": "#f0a500", "catboost": RED}

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import train
from src.features import HORIZONS

## 1. Сургалт ба сонголтын өгөгдөл

- **Сургалт:** train хуваалтын **2023-07-01-ээс** хойшх мөр (хурдны тулд). **(в) бүлэг орно** (ADR 0009): тэдгээрийн train үеийн мөр нь
  оргилоос өмнөх нам гүм үе тул ирээдүйн мэдээлэл агуулахгүй.
- **Хилийн purge:** t+3 нь хуваалтаас гарсан мөрийг `features.py` аль хэдийн хассан (ADR 0009).
- **Valid (early stopping + сонголт):** **(в)-гүй**. (в) нь valid үед оргилсон учраас сонгогдсон сэдэв, түүнийг оруулбал
  хэт "үсрэлттэй" хугацаан дээр сонголт хийгдэнэ (ADR 0005).

In [ ]:
feats = pd.read_parquet(ROOT / "data/processed/features.parquet")
feats = feats[feats["split"] != "test"]                 # test-ийг уншаад л хаяна
groups = train.load_groups()
rs = train.row_sets(feats, groups)
g = feats["article"].map(groups)
pd.DataFrame({
    "сургалт (train ≥ 2023-07-01)": g[rs["train"]].value_counts(),
    "сонголт (valid, (в)-гүй)": g[rs["valid"]].value_counts(),
}).reindex(["а", "б", "в"]).fillna(0).astype(int).rename_axis("бүлэг")

## 2. Optuna: hyperparameter-ийг автоматаар хайх

Hyperparameter (модны гүн, learning rate, regularization...) нь model **өөрөө сурдаггүй**, бид тохируулдаг тоонууд.
Гараар таах оронд **Optuna** ашиглана:

- **TPE sampler** — өмнөх trial-уудаас "аль бүс сайн байсан бэ"-г сурч, дараагийн оролдлогыг тэнд ойртуулна (санамсаргүй хайлтаас ухаалаг).
- **MedianPruner** — trial 10 iteration тутам valid MAE-ээ мэдээлнэ. Өмнөх trial-уудын медианаас муу явж байвал **эрт зогсооно** (цаг хэмнэнэ).
- **Early stopping 100** — valid MAE 100 iteration сайжрахгүй бол зогсож, хамгийн сайн iteration-ыг үлдээнэ.

Алгоритм × horizon бүрт **25 trial** = 3 × 3 × 25 = **225 trial**. h бүрийг **тусад нь** тааруулсан (h1-ийн параметрийг дахин ашиглаагүй).

> ⏱ Энэ cell ~1 цаг ажиллана.

In [ ]:
res = train.run()
table, studies, winner = res["table"], res["studies"], res["winner"]

### Study: шилдэг параметрүүд

Алгоритм × horizon бүрийн хамгийн сайн trial (valid MAE нь (в)-гүй):

In [ ]:
rows = []
for (algo, h), st in studies.items():
    states = pd.Series([t.state.name for t in st.trials]).value_counts()
    rows.append({"algo": algo, "h": h, "valid MAE": round(st.best_value, 4), "шилдэг trial": st.best_trial.number,
                 "дууссан": int(states.get("COMPLETE", 0)), "pruned": int(states.get("PRUNED", 0)),
                 **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in st.best_params.items()}})
pd.DataFrame(rows).set_index(["algo", "h"])

### Study: ялагчийн h3 trial бүр

Trial бүр ямар параметр туршиж, ямар үр дүн гаргасныг харуулна. `PRUNED` нь MedianPruner эрт зогсоосон trial.

In [ ]:
st = studies[(winner, 3)]
df = st.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
df.columns = [c.replace("params_", "").replace("user_attrs_", "") for c in df.columns]
df.round(4)

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{a}" for a in train.ALGOS], shared_yaxes=False)
for i, algo in enumerate(train.ALGOS, 1):
    st = studies[(algo, 3)]
    done = [t for t in st.trials if t.value is not None]
    x, y = [t.number for t in done], [t.value for t in done]
    fig.add_scatter(x=x, y=y, mode="markers", marker_color=GRAY, name="trial", showlegend=i == 1, row=1, col=i)
    fig.add_scatter(x=x, y=np.minimum.accumulate(y), mode="lines", line_color=COLORS[algo], name="хамгийн сайн", showlegend=i == 1, row=1, col=i)
fig.update_layout(title="Optuna-ийн түүх (h3): trial бүрийн valid MAE ба тэр хүртэлх хамгийн сайн", height=380)
fig.update_xaxes(title="trial")
fig.show()

## 3. Сонголт: гурван алгоритмын train / valid MAE

- **valid MAE** = (в)-гүй valid (сонголтын шалгуур). **valid (бүгд)** = (в)-тэй, зөвхөн мэдээлэлд.
- **naive** = "өнөөдрийнх шиг" (Δ̂ = 0). **skill** = 1 − MAE / MAE_naive.
- **train − valid** нь overfit-ийн дохио: train хэт бага, valid өндөр бол model цээжилсэн гэсэн үг.

In [ ]:
t = table.copy()
t["skill"] = 1 - t["valid_mae"] / t["naive_valid_mae"]
t["train − valid"] = t["train_mae"] - t["valid_mae"]
show = t.set_index(["algo", "h"])[["train_mae", "valid_mae", "valid_mae_all", "naive_valid_mae", "skill", "train − valid", "best_iteration"]]
show.columns = ["train MAE", "valid MAE", "valid (бүгд)", "naive", "skill", "train − valid", "iteration"]
show.style.format({c: "{:.4f}" for c in show.columns[:4]} | {"skill": "{:.1%}", "train − valid": "{:+.4f}"})

In [ ]:
avg = table.groupby("algo")[["train_mae", "valid_mae"]].mean().sort_values("valid_mae")
avg.columns = ["train MAE (дундаж)", "valid MAE (дундаж)"]
print(f"Ялагч (h1–h3-ын дундаж valid MAE хамгийн бага): {winner}")
avg.round(4)

## 4. Overfit: ялагчийн learning curve

Iteration (мод) нэмэгдэх тусам train ба valid MAE хэрхэн өөрчлөгдсөнийг харна.
Valid муудаж эхэлвэл (эсвэл 100 iteration сайжрахгүй бол) early stopping зогсооно — **босоо зураас** нь сонгосон iteration.

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"h{h}" for h in HORIZONS])
for i, h in enumerate(HORIZONS, 1):
    f = res["fitted"][(winner, h)]
    it = np.arange(1, len(f.curve["valid"]) + 1)
    fig.add_scatter(x=it, y=f.curve["train"], name="train", line_color=GRAY, showlegend=i == 1, row=1, col=i)
    fig.add_scatter(x=it, y=f.curve["valid"], name="valid ((в)-гүй)", line_color=COLORS[winner], showlegend=i == 1, row=1, col=i)
    fig.add_vline(x=f.best_iteration, line_dash="dot", line_color=RED, row=1, col=i)
fig.update_layout(title=f"{winner}: learning curve (MAE, Δ)", height=380)
fig.update_xaxes(title="iteration")
fig.show()

## 5. Аль feature хамгийн чухал вэ?

Ялагчийн importance-ийг (gain) h1–h3-аар дундажлав.

In [ ]:
imp = pd.DataFrame(res["meta"]["importance_top10"]).sort_values("share")
fig = px.bar(imp, x="share", y="feature", orientation="h", color_discrete_sequence=[COLORS[winner]],
             title=f"{winner}: importance top-10 (h1–h3 дундаж)")
fig.update_layout(xaxis_tickformat=".0%", xaxis_title=None, yaxis_title=None, height=420)
fig.show()

## 6. Экспорт: `app/model/`

Ялагчийн 3 model (`h1`, `h2`, `h3`) + `meta.json` (algo, 18 feature, learning curve, importance top-10, сургасан огноо).
Вэб (ADR 0008) яг **энэ** model-ийг ачаална. Шалгалт: файлаас ачаалсан model ижил таамаг гаргах ёстой.

In [ ]:
m = train.load()
va = res["valid"]
for h in HORIZONS:
    same = np.array_equal(m.predict(va, h), train.predict(res["fitted"][(winner, h)], va[res["features"]]))
    print(f"h{h}: {m.meta['files'][f'h{h}']:8s} ачаалсан = сургасан: {same}")
{k: m.meta[k] for k in ["algo", "trained_at", "train_from", "train_rows", "valid_rows", "best_iteration"]}

## ✅ Дүгнэлт

_(сургалтын дараа бөглөнө)_